# Refinement & Vault Export - Knowledge Graph Quality Tools

Demonstrates the refinement pipeline (entity merging, description enrichment,
relationship inference), typed relations with confidence scoring, and
Obsidian vault export.

**Run:**
```bash
dotenvx run -- uv run jupyter nbconvert --to notebook --execute --inplace notebooks/05_refinement_and_vault.ipynb
```


In [1]:
import json
from pathlib import Path

WORKING_DIR = Path("./_cache/05_refinement_vault")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
VAULT_DIR = WORKING_DIR / "vault_export"
VAULT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working dir: {WORKING_DIR}\nVault dir:   {VAULT_DIR}")


Working dir: _cache/05_refinement_vault
Vault dir:   _cache/05_refinement_vault/vault_export


## Imports + typed relation vocabulary

In [2]:
from nano_graphrag import (
    RELATION_ALIASES, RELATION_VOCABULARY,
    GraphRAG, QueryParam, normalize_relation_type,
)
print(f"Relation types: {len(RELATION_VOCABULARY)}")
print(f"Aliases:         {len(RELATION_ALIASES)}")
print(f"normalize_relation_type('works_for') -> {normalize_relation_type('works_for')!r}")


Relation types: 41
Aliases:         17
normalize_relation_type('works_for') -> 'employed_by'


## Initialize GraphRAG with refinement + vault export

In [3]:
rag = GraphRAG(
    working_dir=str(WORKING_DIR),
    enable_llm_cache=True,
    # --- Refinement pipeline ---
    enable_refinement=True,
    refinement_merge_threshold=0.93,
    refinement_enrich_min_chars=80,
    refinement_infer_confidence=0.80,
    refinement_infer_hub_cap=3,
    refinement_batch_size=50,
    relationship_confidence_threshold=0.0,
    # --- Vault export ---
    vault_path=str(VAULT_DIR),
    vault_export_communities=True,
)
print(f"Refinement: enabled={rag.enable_refinement}")
print(f"  merge_threshold:     {rag.refinement_merge_threshold}")
print(f"  enrich_min_chars:    {rag.refinement_enrich_min_chars}")
print(f"  infer_confidence:    {rag.refinement_infer_confidence}")
print(f"  infer_hub_cap:       {rag.refinement_infer_hub_cap}")
print(f"Vault: path={rag.vault_path}")


2026-05-17T12:04:57.260671Z [info     ] tokenizer_loading              [nano-graphrag] model_name=gpt-4o tokenizer_type=tiktoken


2026-05-17T12:04:57.379735Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=llm_response_cache


2026-05-17T12:04:57.380369Z [info     ] litellm_configured             [nano-graphrag] api_base=None model=openrouter/google/gemma-4-31b-it structured_output=True


2026-05-17T12:04:57.382702Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=full_docs


2026-05-17T12:04:57.383813Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=text_chunks


...

2026-05-17T12:04:57.385766Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=document_index


2026-05-17T12:04:57.386798Z [info     ] sqlite_kv_loaded               [nano-graphrag] entries=0 namespace=graph_contribution_index


2026-05-17T12:04:57.391571Z [info     ] hnsw_index_created             [nano-graphrag] namespace=entities


2026-05-17T12:04:57.392534Z [info     ] entity_registry_initialized    [nano-graphrag]


Refinement: enabled=True
  merge_threshold:     0.93
  enrich_min_chars:    80
  infer_confidence:    0.8
  infer_hub_cap:       3
Vault: path=_cache/05_refinement_vault/vault_export


## Insert sample docs - overlapping content triggers merge + infer

In [4]:

import textwrap
docs = {
    "einstein_bio": textwrap.dedent("""\
        Albert Einstein was a German-born theoretical physicist who developed the
        theory of relativity, one of the two pillars of modern physics. His work
        is known for its influence on the philosophy of science. Einstein
        published more than 300 scientific papers and developed the general
        theory of relativity in 1915."""),
    "einstein_later": textwrap.dedent("""\
        Einstein worked at the Institute for Advanced Study in Princeton from
        1933 until his death in 1955. He collaborated with Nathan Rosen on the
        Einstein-Rosen bridge (wormhole theory) in 1935."""),
    "relativity_theory": textwrap.dedent("""\
        The Theory of Relativity, pioneered by Albert Einstein, revolutionized
        physics in the early 20th century. The special theory (1905) and general
        theory (1915) changed our understanding of space, time, and gravity."""),
}
print("Inserting documents...")
await rag.ainsert_documents(docs)
print("Done.\n")


2026-05-17T12:04:57.396060Z [info     ] delta_detection                [nano-graphrag] changed_docs=0 run_id=22b14a94 total_docs=3


2026-05-17T12:04:57.401188Z [info     ] extraction_start               [nano-graphrag] concurrency=4 flush_every=50 run_id=22b14a94 total_docs=3


Inserting documents...

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



2026-05-17T12:05:51.816748Z [info     ] llm_call_complete              [nano-graphrag] completion_tokens=483 cost_usd=0.0 latency_ms=54407.5 model=openrouter/google/gemma-4-31b-it prompt_tokens=352 run_id=22b14a94 total_tokens=835


2026-05-17T12:05:51.819299Z [info     ] extraction_chunk_progress      [nano-graphrag] elapsed_s=0.0 entities=4 pct=100 processed=1 relations=3 run_id=22b14a94 total=1


...

2026-05-17T12:07:04.244930Z [info     ] llm_call_complete              [nano-graphrag] completion_tokens=301 cost_usd=0.0 latency_ms=11664.3 model=openrouter/google/gemma-4-31b-it prompt_tokens=1450 run_id=22b14a94 total_tokens=1751



Provider List: https://docs.litellm.ai/docs/providers



2026-05-17T12:07:11.765457Z [info     ] llm_call_complete              [nano-graphrag] completion_tokens=410 cost_usd=0.0 latency_ms=19186.4 model=openrouter/google/gemma-4-31b-it prompt_tokens=1661 run_id=22b14a94 total_tokens=2071


2026-05-17T12:07:11.841629Z [info     ] graph_write                    [nano-graphrag] edges=13 nodes=12 run_id=22b14a94



Provider List: https://docs.litellm.ai/docs/providers

Done.



## Query BEFORE refinement - baseline

In [5]:
print("=== BEFORE REFINEMENT ===")
result = await rag.aquery(
    "Who did Einstein collaborate with and where did he work?",
    param=QueryParam(mode="local"),
)
print(result)


2026-05-17T12:07:11.852736Z [info     ] query_start                    [nano-graphrag] mode=local query='Who did Einstein collaborate with and where did he work?' run_id=59e2a480


=== BEFORE REFINEMENT ===


2026-05-17T12:07:30.565275Z [info     ] embedding_call_complete        [nano-graphrag] completion_tokens=0 cost_usd=0.0 latency_ms=18711.7 mode=local model=openrouter/qwen/qwen3-embedding-8b num_texts=1 prompt_tokens=12 run_id=59e2a480 total_tokens=12


2026-05-17T12:07:30.571897Z [info     ] local_query_context            [nano-graphrag] communities=7 entities=12 mode=local relations=12 run_id=59e2a480 text_units=3



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



2026-05-17T12:07:38.713195Z [info     ] llm_call_complete              [nano-graphrag] completion_tokens=17 cost_usd=0.0 latency_ms=8139.8 mode=local model=openrouter/google/gemma-4-31b-it prompt_tokens=3466 run_id=59e2a480 total_tokens=3483


2026-05-17T12:07:38.715962Z [info     ] query_complete                 [nano-graphrag] answer_chars=100 latency_ms=26862.5 mode=local run_id=59e2a480



Provider List: https://docs.litellm.ai/docs/providers

Einstein collaborated with Nathan Rosen and worked at the Institute for Advanced Study in Princeton.


## Run refinement pipeline (merge → enrich → infer)

In [6]:
# Pipeline phases:
#   merge  - consolidates near-duplicate entities (e.g. "EINSTEIN" + "Albert Einstein")
#   enrich - expands thin descriptions (skips those already >80 chars)
#   infer  - discovers new relationships between co-occurring entities

print("Running refinement pipeline...")
results = await rag.arefine()
for phase, stats in results.items():
    print(f"\n  {phase}:")
    for k, v in stats.items():
        print(f"    {k}: {v}")


2026-05-17T12:07:38.728471Z [info     ] refinement_merge_start         [nano-graphrag]


Running refinement pipeline...


2026-05-17T12:07:40.482926Z [info     ] embedding_call_complete        [nano-graphrag] completion_tokens=0 cost_usd=0.0 latency_ms=1754.0 model=openrouter/qwen/qwen3-embedding-8b num_texts=12 prompt_tokens=180 total_tokens=180


2026-05-17T12:07:40.485616Z [info     ] refinement_enrich_start        [nano-graphrag]


2026-05-17T12:07:40.485905Z [info     ] refinement_enrich_start        [nano-graphrag] thin_count=8


2026-05-17T12:07:40.486580Z [info     ] refinement_enrich_done         [nano-graphrag] enriched=0 examined=8 skipped=8 validation_failed=0


2026-05-17T12:07:40.486950Z [info     ] refinement_infer_start         [nano-graphrag]



  merge:
    examined: 0
    merged: 0
    skipped: 0

  enrich:
    examined: 8
    enriched: 0
    skipped: 8
    validation_failed: 0

  infer:
    examined: 0
    inferred: 0
    rejected_by_llm: 0
    rejected_by_cache: 0

Provider List: https://docs.litellm.ai/docs/providers



## Query AFTER refinement - richer connections

In [7]:
print("=== AFTER REFINEMENT ===")
result = await rag.aquery(
    "Who did Einstein collaborate with and where did he work?",
    param=QueryParam(mode="local"),
)
print(result)


2026-05-17T12:07:40.493648Z [info     ] query_start                    [nano-graphrag] mode=local query='Who did Einstein collaborate with and where did he work?' run_id=e3a482b3


=== AFTER REFINEMENT ===


2026-05-17T12:07:50.959660Z [info     ] embedding_call_complete        [nano-graphrag] completion_tokens=0 cost_usd=0.0 latency_ms=10465.5 mode=local model=openrouter/qwen/qwen3-embedding-8b num_texts=1 prompt_tokens=12 run_id=e3a482b3 total_tokens=12


2026-05-17T12:07:50.964290Z [info     ] local_query_context            [nano-graphrag] communities=7 entities=12 mode=local relations=12 run_id=e3a482b3 text_units=3


2026-05-17T12:07:50.965580Z [info     ] llm_cache_hit                  [nano-graphrag] args_hash=b81f6b7db3496484d66165471fbefbf8 mode=local model=openrouter/google/gemma-4-31b-it run_id=e3a482b3


2026-05-17T12:07:50.965985Z [info     ] query_complete                 [nano-graphrag] answer_chars=100 latency_ms=10472.1 mode=local run_id=e3a482b3



Provider List: https://docs.litellm.ai/docs/providers

Einstein collaborated with Nathan Rosen and worked at the Institute for Advanced Study in Princeton.


## Typed Relations - vocabulary and normalization

In [8]:
print(f"Full vocabulary ({len(RELATION_VOCABULARY)} types):")
print(f"  {', '.join(sorted(RELATION_VOCABULARY))}")

test_types = ["works_for", "built_on", "associated_with", "CREATED_BY", "authored", "RandomType"]
print("\nNormalization examples:")
for t in test_types:
    print(f"  {t:25s} -> {normalize_relation_type(t)}")


Full vocabulary (41 types):
  authored_by, builds_on, causes, cites, classified_as, collaborates_with, competes_with, consumes, contains, contradicts, created_by, depends_on, developed_by, employed_by, enables, extends, founded_by, has_characteristic, headquartered_in, implements, influences, instance_of, invests_in, led_by, located_in, managed_by, member_of, operates_in, originates_from, parent_organization_of, part_of, precedes, prevents, produces, provides, references, related_to, subsidiary_of, supports, uses, works_on

Normalization examples:
  works_for                 -> employed_by
  built_on                  -> builds_on
  associated_with           -> related_to
  CREATED_BY                -> created_by
  authored                  -> authored_by
  RandomType                -> related_to


## Export Obsidian Vault

In [9]:
print("Exporting vault...")
await rag.aexport_vault()
md_files = sorted(VAULT_DIR.rglob("*.md"))
print(f"Exported {len(md_files)} markdown files")
for f in md_files[:10]:
    print(f"  {f.relative_to(VAULT_DIR)}")
if len(md_files) > 10:
    print(f"  ... and {len(md_files) - 10} more")

# Sample one entity file
for f in md_files:
    if f.name != "_index.md" and "community" not in f.stem:
        content = f.read_text().strip()
        print(f"\nSample entity ({f.name}):\n{content[:400]}...")
        break


2026-05-17T12:07:50.983542Z [info     ] vault_export_done              [nano-graphrag] communities=7 entities=12 path=_cache/05_refinement_vault/vault_export sparse_rolled_up=0


Exporting vault...
Exported 25 markdown files
  communities/Albert Einstein and Germany.md
  communities/Albert Einstein and the General Theory of Relativity.md
  communities/Albert Einstein and the Theory of Relativity.md
  communities/General Theory of Relativity.md
  communities/Theory of Relativity and Philosophy of Science.md
  communities/Theory of Relativity and its Components.md
  communities/_index.md
  entities/EVENT/EINSTEIN-ROSEN BRIDGE.md
  entities/EVENT/GENERAL THEORY OF RELATIVITY.md
  entities/EVENT/GENERAL THEORY.md
  ... and 15 more

Sample entity (Albert Einstein and Germany.md):
---
id: l0_c1
title: Albert Einstein and Germany
rating: 0.00
---
# Albert Einstein and Germany

# Albert Einstein and Germany

The community consists of Albert Einstein, a German-born theoretical physicist, and the country of Germany. The relationship is centered on Einstein's birth in Germany and his pioneering work on the Theory of Relativity.

## Albert Einstein's scientific contributio